# 1) LOAD TEST SET

In [2]:
import tensorflow as tf
from tensorflow.keras import layers, Model, optimizers
from Autoencoder_training import Autoencoder


IMG_HEIGHT = 224
IMG_WIDTH  = 224
AUTOTUNE   = tf.data.AUTOTUNE
DATASET_BASE = '../../data'         
OBJ           = 'leather'
MODEL_PATH    = '/Users/vegardhaugerud/mvtec_anomaly_detection_project/notebooks/Autoencoder/checkpoints/autoencoder_01_val5755.8306.keras'        
OUT_BASE      = '../../output/anomaly_maps'                  
IMG_SIZE      = (224, 224)

model = tf.keras.models.load_model(MODEL_PATH)
normalize = tf.keras.layers.Rescaling(1./255)


ModuleNotFoundError: No module named 'Autoencoder_training'

In [7]:
raw_test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    '../../data/leather/test',
    labels='inferred',
    label_mode='int',            # 0 or 1 per subfolder
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=1,
    shuffle=False
)
class_names = raw_test_ds.class_names
print("class_names:", class_names)  

normalize = tf.keras.layers.Rescaling(1./255)
test_ds = (
    raw_test_ds
     .map(lambda x,y: (normalize(x), y), num_parallel_calls=AUTOTUNE)
)

Found 124 files belonging to 6 classes.


class_names: ['color', 'cut', 'fold', 'glue', 'good', 'poke']


# 2) NORMALIZE


In [ ]:
normalize = tf.keras.layers.Rescaling(1.0/255)

test_ds = test_ds.map(lambda x, y: (normalize(x), y),num_parallel_calls=AUTOTUNE)


# 3) INFERENCE & ERROR COMPUTATION


In [ ]:
import numpy as np
from tqdm import tqdm 

errors = []
labels = []

for x_batch, y_batch in tqdm(test_ds, total=len(raw_test_ds)):
    x_rec = model(x_batch, training=False)
    mse = tf.reduce_mean(tf.square(x_batch - x_rec), axis=[1,2,3])
    errors.append(mse.numpy()[0])
    labels.append(int(y_batch.numpy()[0]))

errors = np.array(errors)
labels = np.array(labels)

print(f"Test set size: {len(errors)} images")


# 4) Creating anomaly maps



In [ ]:
# gen_maps.py
import os
import numpy as np
from PIL import Image
import tifffile
import tensorflow as tf


test_root = os.path.join(DATASET_BASE, OBJ, 'test')
for defect in os.listdir(test_root):            # defect = 'good' or 'hole'
    src_dir = os.path.join(test_root, defect)
    dst_dir = os.path.join(OUT_BASE, OBJ, 'test', defect)
    os.makedirs(dst_dir, exist_ok=True)

    for fname in os.listdir(src_dir):
        if not fname.lower().endswith('.png'):
            continue

        # load & preprocess
        img = Image.open(os.path.join(src_dir, fname)).convert('RGB')
        img = img.resize(IMG_SIZE)
        arr = np.array(img, dtype=np.float32)   # H×W×3, [0–255]
        arr = arr[None, ...]  # add batch dimension
        arr = normalize(arr)                    # → [0–1]

        # reconstruct & compute per-pixel MSE
        rec = model(arr, training=False).numpy()[0]  
        amap = np.mean((arr[0] - rec)**2, axis=-1)  # H×W

        # save as float32 TIFF
        out_path = os.path.join(dst_dir, fname.replace('.png','') + '.tiff')
        tifffile.imwrite(out_path, amap.astype(np.float32))

print("Done generating anomaly maps.")


# Running eval function


In [ ]:
import subprocess

def run_leather_evaluation():
    cmd = [
        "python", "../evaluate/patchCore_Eval/evaluate_experiment2.py",
        "--dataset_base_dir", "../data",
        "--anomaly_maps_dir", "../output/anomaly_maps",
        "--output_dir", "../output/metrics",
        "--evaluated_objects", "leather",
        "--pro_integration_limit", "0.3"
    ]
    subprocess.run(cmd, check=True)

if __name__ == "__main__":
    run_leather_evaluation()
    print("Evaluation completed successfully.")